Detta är koden till chatbotten som används i inlämningsuppgiften

## Importer

In [57]:
from pypdf import PdfReader
from pathlib import Path
import re
from google import genai
import os
from google.genai import types
from dotenv import load_dotenv
import json
import time
import numpy as np


In [55]:
load_dotenv()

True

In [29]:
client = genai.Client(api_key=os.getenv("API_KEY"))

# RAG-delen

## Inläsning av dokumenten

In [45]:
# Läser in alla pdferna som ligger i mappen "Dokument till RAG" i mitt repository
folder_path = Path("Dokumenten till RAG")
pdf_files = list(folder_path.glob("*.pdf"))

documents = []

for pdf in folder_path.glob("*.pdf"): #läser in alla alla dokument som slutar med pdf
    reader = PdfReader(pdf)

    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        # Lägger bara till om det verkligen finns någon text på sidan
        if page_text:
            text += page_text + "\n"
    
    # sparar från vilket dokument texten kommer ifrån
    documents.append({
        "source": pdf.name,
        "text": text
    })


## Första chunkningen

In [50]:
chunks = []
n = 10
overlap = 2

for doc in documents: 
    sentences = re.split(r'(?<=[.!?]) +', doc["text"])

    for i in range(0, len(sentences), n - overlap):
        chunk_sentences = sentences[i:i+n]

        chunk_text = " ".join(chunk_sentences)

        chunks.append({
            "text": chunk_text,
            "source": doc["source"]
        })

In [51]:
print(len(chunks))

215


## Embeddings

In [52]:
def create_embeddings(text,
                      model="gemini-embedding-001",
                      task_type="SEMANTIC_SIMILARITY"):
    return client.models.embed_content(
        model=model,
        contents=text,
        config=types.EmbedContentConfig(task_type=task_type)
    )

Eftersom jag fick problem med att jag överskred min quota så har jag minskat antal manualer från 5 till 1 samt satt in att spara embeddnings mellan varje lyckad embedding för att inte behöva köra om från början om jag uppnår min quota. Jag satt även in en liten fördröjning mellan chunksen för att minska belastningen

In [ ]:
save_path = "embeddings.json"

# ladda in redan sparade embeddings om filen finns
if os.path.exists(save_path):
    with open(save_path, "r", encoding="utf-8") as f:
        embeddings = json.load(f)

else:
    embeddings = []

already_done = {item["text"] for item in embeddings}

for chunk in chunks:
    if chunk["text"] in already_done:
        continue

    try:
        response = create_embeddings(chunk["text"])
    
        item = {
        "text": chunk["text"],
        "source": chunk["source"],
        "embedding": response.embeddings[0].values
        }

        embeddings.append(item)

        # spara direkt efter varje lyckad embedding
        with open(save_path, "w", encoding="utf-8") as f:
            json.dump(embeddings, f, ensure_ascii=False, indent=2)

        print(f"Sparade chunk från {chunk['source']}")

        time.sleep(1) # liten paus för att minska belastningen

    except Exception as e:
        print("Stopp:", e)
        break

Sparade chunk från Electrolux_tvättmaskinsmodeller.pdf
Sparade chunk från EW2F3148P1_914912588_Produktinformationsblad.pdf
Sparade chunk från EW2F3148P1_UserManual.pdf
Sparade chunk från EW2F3148P1_UserManual.pdf
Sparade chunk från EW2F3148P1_UserManual.pdf
Sparade chunk från EW2F3148P1_UserManual.pdf
Sparade chunk från EW2F3148P1_UserManual.pdf
Sparade chunk från EW2F3148P1_UserManual.pdf
Sparade chunk från EW2F3148P1_UserManual.pdf
Sparade chunk från EW2F3148P1_UserManual.pdf
Sparade chunk från EW2F3148P1_UserManual.pdf
Sparade chunk från EW2F3148P1_UserManual.pdf
Sparade chunk från EW2F3148P1_UserManual.pdf
Sparade chunk från EW2F3148P1_UserManual.pdf
Sparade chunk från EW2F3148P1_UserManual.pdf
Sparade chunk från EW2F3148P1_UserManual.pdf
Sparade chunk från EW2F3148P1_UserManual.pdf
Sparade chunk från EW2F3148P1_UserManual.pdf
Sparade chunk från EW2F3148P1_UserManual.pdf
Sparade chunk från EW2F3148P1_UserManual.pdf
Sparade chunk från EW2F3148P1_UserManual.pdf
Sparade chunk från EW2

## Semantisk sökning

In [58]:
def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))